In [2]:
# basic imports
import pandas as pd
import sys
import numpy as np
import pandas as pd
import scanpy as sc

# add `Tangram` to path
import tangram as tg
# datanumber = '7'
# sc_file_path = f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{datanumber}/scRNA.h5ad"
# spatial_file_path = f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{datanumber}/Spatial.h5ad"
# celltype_key = "cell_type"
# output_file_path = f"../Data{datanumber}/tangram_composition.csv"

sc_file_path = "/mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_SC.h5ad"
spatial_file_path = "/mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_ST.h5ad"
celltype_key = "cell_type"
output_file_path = "./tangram_composition.csv"



ad_sc = sc.read_h5ad(sc_file_path)
ad_sp = sc.read_h5ad(spatial_file_path)

# use raw count both of scrna and spatial
sc.pp.normalize_total(ad_sc)
celltype_counts = ad_sc.obs[celltype_key].value_counts()
celltype_drop = celltype_counts.index[celltype_counts < 2]
print(f'Drop celltype {list(celltype_drop)} contain less 2 sample')
ad_sc = ad_sc[~ad_sc.obs[celltype_key].isin(celltype_drop),].copy()
sc.tl.rank_genes_groups(ad_sc, groupby=celltype_key, use_raw=False)
markers_df = pd.DataFrame(ad_sc.uns["rank_genes_groups"]["names"]).iloc[0:200, :]
print(markers_df)
genes_sc = np.unique(markers_df.melt().value.values)
print(genes_sc)
genes_st = ad_sp.var_names.values
genes = list(set(genes_sc).intersection(set(genes_st)))

tg.pp_adatas(ad_sc, ad_sp, genes=genes)

ad_map = tg.map_cells_to_space(
                   ad_sc,
                   ad_sp,
                   mode='clusters',
                   cluster_label=celltype_key)

tg.project_cell_annotations(ad_map, ad_sp, annotation=celltype_key)

# 从映射矩阵提取细胞类型组成
# ad_map.X 是 [n_clusters, n_spots] 的矩阵，表示每个簇在每个 spot 的概率/比例
# ad_map.obs 中直接包含 cell_type 列

# 直接使用 ad_map.obs 中的 cell_type
celltype_names = ad_map.obs['cell_type'].tolist()
print(f"Cell types: {celltype_names}")

# 转置以获得 [n_spots, n_celltypes] 的组成矩阵
cell_composition_tangram = pd.DataFrame(
    ad_map.X.T,  # 转置
    index=ad_sp.obs_names,  # spot 名称
    columns=celltype_names  # celltype 名称
)

print(f"\nCell composition matrix shape: {cell_composition_tangram.shape}")
print(f"Column names: {cell_composition_tangram.columns.tolist()}")
print(f"\nFirst few rows:\n{cell_composition_tangram.head()}")

# 验证每行和（应该接近 1）
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\nRow sums - min: {row_sums.min():.4f}, max: {row_sums.max():.4f}, mean: {row_sums.mean():.4f}")

# 保存结果
# 🔧 归一化 Tangram 结果
print("\n" + "="*60)
print("NORMALIZING TANGRAM RESULTS")
print("="*60)

# 检查当前的行和
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\n❌ BEFORE normalization:")
print(f"   Row sums - min: {row_sums.min():.6f}, max: {row_sums.max():.6f}, mean: {row_sums.mean():.6f}")
print(f"   Example: first row sum = {row_sums.iloc[0]:.6f}")

# 归一化：每行除以该行的和
cell_composition_tangram = cell_composition_tangram.div(row_sums, axis=0)

# 验证归一化后
row_sums_after = cell_composition_tangram.sum(axis=1)
print(f"\n✅ AFTER normalization:")
print(f"   Row sums - min: {row_sums_after.min():.6f}, max: {row_sums_after.max():.6f}, mean: {row_sums_after.mean():.6f}")
print(f"   Example: first row sum = {row_sums_after.iloc[0]:.6f}")

print(f"\nFirst row values (now as percentages):")
print(cell_composition_tangram.head())

# 重新保存归一化后的结果
cell_composition_tangram.to_csv(output_file_path)
print(f"\n💾 Saved normalized results!")
print("="*60)


Drop celltype [] contain less 2 sample
        ASDC  B Cell      CD1C    CLEC9A Endothelial Cell Epithelial  \
0       CFL1   RPL11      CD74    TMSB4X              VIM       PERP   
1     TMSB10   RPL10    TMSB4X      CD74            ECSCR      FXYD3   
2      COTL1   RPL32  HLA-DPB1  HLA-DPA1            GNG11       KRT5   
3     CORO1A   RPS14   HLA-DRA    TMSB10           IGFBP7        SFN   
4       SPI1   RPL15  HLA-DPA1  HLA-DPB1           IFITM3      KRT14   
..       ...     ...       ...       ...              ...        ...   
195  LAMTOR1    GNB2     NABP1  SERPINF2            PTPRB      GIPC1   
196  FAM110A   RPS17   NCKAP1L     GNAI2         SERPING1      ESRP1   
197  SUPT4H1  GLT1D1      EMP3  SERPINF1           TRIOBP     DHCR24   
198   CLEC4A   MBNL1      CTSS     PDIA3             GUK1      GSTM3   
199   MIR142  NAP1L1       GCA       DSE            TACC1     ZNF706   

    Fibroblast        LC      MDSC      Mac Melanocyte Multiplet        NK  \
0         CD63    

INFO:root:1669 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:15780 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 1669 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.358, KL reg: 0.677
Score: 0.684, KL reg: 0.001
Score: 0.686, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001
Score: 0.687, KL reg: 0.001


INFO:root:Saving results..
INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


Cell types: ['Epithelial', 'LC', 'CD1C', 'Mac', 'Melanocyte', 'Tcell', 'Endothelial Cell', 'PDC', 'Multiplet', 'CLEC9A', 'ASDC', 'Fibroblast', 'MDSC', 'NK', 'B Cell']

Cell composition matrix shape: (646, 15)
Column names: ['Epithelial', 'LC', 'CD1C', 'Mac', 'Melanocyte', 'Tcell', 'Endothelial Cell', 'PDC', 'Multiplet', 'CLEC9A', 'ASDC', 'Fibroblast', 'MDSC', 'NK', 'B Cell']

First few rows:
       Epithelial        LC      CD1C           Mac  Melanocyte     Tcell  \
10x16    0.000466  0.000560  0.001053  7.893439e-07    0.000364  0.000064   
10x18    0.002760  0.001356  0.000884  2.429927e-05    0.001409  0.000009   
10x20    0.002801  0.002855  0.002468  1.338973e-03    0.000607  0.001742   
10x22    0.006019  0.004588  0.007587  6.930153e-04    0.000127  0.005070   
10x24    0.000013  0.002372  0.000948  5.747350e-03    0.000076  0.002284   

       Endothelial Cell       PDC  Multiplet    CLEC9A      ASDC  Fibroblast  \
10x16          0.000010  0.000050   0.000012  0.000077  0.0000

In [3]:
ad_ge = tg.project_genes(
    adata_map=ad_map,          # 上一步生成的映射结果
    adata_sc=ad_sc,     # 单细胞数据源
    cluster_label=celltype_key # 必须与 map_cells_to_space 中的 cluster_label 一致
)

print("\nGene projection completed!")
df_expression = ad_ge.to_df()
df_expression.columns = df_expression.columns.str.upper()

# 保存为 CSV
df_expression.to_csv("trangram_expression.csv")


Gene projection completed!
